In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json

import numpy as np
import pandas as pd
from scipy import stats

TASK_DIR = Path.cwd().parent
INPUT_DIR = TASK_DIR / "input"
OUTPUT_DIR = TASK_DIR / "output"

LONG_PATH = INPUT_DIR / "productivity_long.csv"
TRAJECTORIES_PATH = INPUT_DIR / "trajectories.npy"
SIM_MANIFEST_PATH = INPUT_DIR / "simulation_manifest.json"

MODEL_NAME = "ARW4"
MODEL_TAG = "arw4"
EPS = .49
Y = 20
STAGES = {"years_1_4": np.arange(1,5),"years_5_7": np.arange(5,8),"years_8_20": np.arange(8,21)}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
long = pd.read_csv(LONG_PATH)
trajs = np.load(TRAJECTORIES_PATH)
emp = long.loc[long.CareerAge.between(0,Y),["dblp_id","CareerAge","pubs_adj"]].rename(columns={"dblp_id": "id"}).copy()
sim = pd.DataFrame({"id": np.repeat(np.arange(trajs.shape[1]),Y + 1),"CareerAge": np.tile(np.arange(Y + 1),trajs.shape[1]),"pubs_adj": trajs.T.ravel()})
emp["source"], sim["source"] = "empirical", "simulated"
all_long = pd.concat([emp,sim],ignore_index=True)

emp_wide = emp.pivot(index="id",columns="CareerAge",values="pubs_adj").reindex(columns=np.arange(Y + 1)).dropna()
emp_trajs = emp_wide.to_numpy().T
print(f"Empirical complete: {emp_trajs.shape[1]:,}")
print(f"Simulated: {trajs.shape[1]:,}")

Empirical complete: 681
Simulated: 10,000


In [3]:
def mode_hist(x, bins=100):
    counts, bars = np.histogram(x,bins=bins)
    i = np.argmax(counts)
    return (bars[i] + bars[i + 1]) / 2

def gini(x):
    x = np.sort(np.asarray(x,dtype=float)); n = len(x)
    return (2 * np.sum((np.arange(n) + 1) * x) / (n * x.sum()) - (n + 1) / n) if x.sum() else 0

def top_share(x, p):
    x = np.sort(np.asarray(x,dtype=float)); k = max(1,int(np.ceil(len(x) * p)))
    return x[-k:].sum() / x.sum() if x.sum() else 0

def ecdf(x):
    x = np.sort(np.asarray(x)); return x, np.arange(1,len(x) + 1) / len(x)

def qq_coords(x):
    x = np.asarray(x); x = x[x > 0]
    z = np.log2(x); z = (z - z.mean()) / z.std(ddof=1); z = np.sort(z)
    return stats.norm.ppf((np.arange(len(z)) + .5) / len(z)), z

def laplace_fit(x):
    x = np.asarray(x); mu = np.median(x); alpha = np.mean(np.abs(x - mu))
    return mu, alpha, stats.kstest(x,"laplace",args=(mu,alpha)).statistic, stats.kstest(x,"laplace",args=(mu,alpha)).pvalue

In [4]:
canonical_rows, yearwise_rows = [], []
for (source,year), d in all_long.groupby(["source","CareerAge"]):
    x = d.pubs_adj.to_numpy(); lx = np.log(x + EPS); pos = x[x > 0]; lpos = np.log(pos + EPS)
    canonical_rows.append({"source": source,"career_age": year,"n": len(x),"mean": x.mean(),"median": np.median(x)})
    yearwise_rows.append({"source": source,"career_age": year,"n": len(x),"frac_zero": np.mean(x == 0),"mean_prod": x.mean(),"median_prod": np.median(x),"sd_prod": x.std(ddof=1),"mean_log_prod": lx.mean(),"var_log_prod": lx.var(ddof=1),"mean_log_prod_pos": lpos.mean(),"var_log_prod_pos": lpos.var(ddof=1),"q25_prod": np.quantile(x,.25),"q50_prod": np.quantile(x,.5),"q75_prod": np.quantile(x,.75),"q90_prod": np.quantile(x,.9),"q95_prod": np.quantile(x,.95)})

canonical = pd.DataFrame(canonical_rows)
yearwise = pd.DataFrame(yearwise_rows)
canonical.to_csv(OUTPUT_DIR / "arw4_canonical_trajectory_stats.csv",index=False)
yearwise.to_csv(OUTPUT_DIR / "arw4_yearwise_productivity_stats.csv",index=False)

In [5]:
def transitions(d):
    d = d.sort_values(["id","CareerAge"]).copy()
    d["next_year"] = d.groupby("id").CareerAge.shift(-1)
    d["pubs_adj_next"] = d.groupby("id").pubs_adj.shift(-1)
    d = d.loc[d.next_year.eq(d.CareerAge + 1)].copy()
    d["destination_year"] = d.CareerAge + 1
    d["raw_delta"] = d.pubs_adj_next - d.pubs_adj
    d["log_delta"] = np.log(d.pubs_adj_next + EPS) - np.log(d.pubs_adj + EPS)
    return d

emp_dx, sim_dx = transitions(emp), transitions(sim)
emp_dx["source"], sim_dx["source"] = "empirical", "simulated"
all_dx = pd.concat([emp_dx,sim_dx],ignore_index=True)

log_rows = []
for (source,year), d in all_dx.groupby(["source","destination_year"]):
    x = d.log_delta.to_numpy()
    log_rows.append({"source": source,"transition_year": year - 1,"destination_year": year,"n": len(x),"mean_log_delta": x.mean(),"median_log_delta": np.median(x),"mode_log_delta": mode_hist(x),"var_log_delta": x.var(ddof=1),"q25_log_delta": np.quantile(x,.25),"q50_log_delta": np.quantile(x,.5),"q75_log_delta": np.quantile(x,.75),"q90_log_delta": np.quantile(x,.9),"q95_log_delta": np.quantile(x,.95)})
pd.DataFrame(log_rows).to_csv(OUTPUT_DIR / "arw4_yearwise_log_delta_stats.csv",index=False)

In [6]:
laplace_values, laplace_rows = [], []
for stage, years in STAGES.items():
    for source, d in all_dx.loc[all_dx.destination_year.isin(years)].groupby("source"):
        x = d.raw_delta.to_numpy(); mu, alpha, ks, p = laplace_fit(x)
        laplace_rows.append({"source": source,"stage": stage,"n": len(x),"mu_hat": mu,"alpha_hat": alpha,"ks_stat": ks,"ks_pvalue": p,"mean": x.mean(),"sd": x.std(ddof=1),"q01": np.quantile(x,.01),"q50": np.quantile(x,.5),"q99": np.quantile(x,.99)})
        laplace_values.extend({"source": source,"stage": stage,"value": v} for v in x)

pd.DataFrame(laplace_rows).to_csv(OUTPUT_DIR / "arw4_stage_raw_increment_laplace.csv",index=False)
pd.DataFrame(laplace_values).to_csv(OUTPUT_DIR / "arw4_stage_raw_increment_values.csv",index=False)

In [7]:
rank_rows = []
for source, arr in [("empirical",emp_trajs),("simulated",trajs)]:
    for year in range(Y + 1): rank_rows.append({"source": source,"career_age": year,"rank_correlation_with_year0": stats.spearmanr(arr[0],arr[year]).statistic})
pd.DataFrame(rank_rows).to_csv(OUTPUT_DIR / "arw4_rank_correlations.csv",index=False)

year_max_rows, cum_y5_rows = [], []
for source, arr in [("empirical",emp_trajs),("simulated",trajs)]:
    year_max_rows.extend({"source": source,"value": v} for v in np.argmax(arr,axis=0))
    cum_y5_rows.extend({"source": source,"value": v} for v in arr[:6].sum(axis=0))
year_max = pd.DataFrame(year_max_rows); cum_y5 = pd.DataFrame(cum_y5_rows)
year_max.to_csv(OUTPUT_DIR / "arw4_year_of_maximum.csv",index=False)
cum_y5.to_csv(OUTPUT_DIR / "arw4_cumulative_through_year5.csv",index=False)

ks_rows = []
for name, d in [("year_of_maximum",year_max),("cumulative_through_year5",cum_y5)]:
    a, b = d.loc[d.source.eq("empirical"),"value"], d.loc[d.source.eq("simulated"),"value"]
    r = stats.ks_2samp(a,b)
    ks_rows.append({"diagnostic": name,"statistic": r.statistic,"pvalue": r.pvalue,"n_empirical": len(a),"n_simulated": len(b)})
pd.DataFrame(ks_rows).to_csv(OUTPUT_DIR / "arw4_ks_diagnostics.csv",index=False)

In [8]:
def empirical_last_four(d):
    values = []
    for _, g in d.groupby("id"):
        g = g.sort_values("CareerAge").copy()
        g["rolling4"] = g.pubs_adj.rolling(4,min_periods=4).sum()
        g["span4"] = g.CareerAge - g.CareerAge.shift(3)
        x = g.loc[g.span4.eq(3),"rolling4"].dropna()
        if len(x): values.append(x.iloc[-1])
    return np.array(values)

aggregates = {"year_20": {"empirical": emp_trajs[20],"simulated": trajs[20]},"last_four": {"empirical": empirical_last_four(emp),"simulated": trajs[-4:].sum(axis=0)},"full_career": {"empirical": emp_trajs.sum(axis=0),"simulated": trajs.sum(axis=0)}}
aggregate_rows, fit_rows, qq_rows, inequality_rows = [], [], [], []
for aggregation, sources in aggregates.items():
    for source, x in sources.items():
        x = np.asarray(x); aggregate_rows.extend({"source": source,"aggregation": aggregation,"value": v} for v in x)
        shape, loc, scale = stats.lognorm.fit(x + EPS,floc=0); ks = stats.kstest(x + EPS,"lognorm",args=(shape,loc,scale))
        fit_rows.append({"source": source,"aggregation": aggregation,"n": len(x),"epsilon_shift": EPS,"shape": shape,"loc": loc,"scale": scale,"ks_stat": ks.statistic,"ks_pvalue": ks.pvalue})
        theoretical, observed = qq_coords(x)
        qq_rows.extend({"source": source,"aggregation": aggregation,"theoretical": a,"observed": b} for a,b in zip(theoretical,observed))
        inequality_rows.extend([{"source": source,"aggregation": aggregation,"metric": "gini","value": gini(x)},{"source": source,"aggregation": aggregation,"metric": "top_1_percent_share","value": top_share(x,.01)},{"source": source,"aggregation": aggregation,"metric": "top_5_percent_share","value": top_share(x,.05)},{"source": source,"aggregation": aggregation,"metric": "top_10_percent_share","value": top_share(x,.10)}])

pd.DataFrame(aggregate_rows).to_csv(OUTPUT_DIR / "arw4_aggregate_productivity_values.csv",index=False)
pd.DataFrame(fit_rows).to_csv(OUTPUT_DIR / "arw4_lognormal_fit_stats.csv",index=False)
pd.DataFrame(qq_rows).to_csv(OUTPUT_DIR / "arw4_lognormal_qq_coordinates.csv",index=False)
pd.DataFrame(inequality_rows).to_csv(OUTPUT_DIR / "arw4_inequality_summary.csv",index=False)

In [9]:
wasserstein = []
for year in range(Y + 1):
    a = emp.loc[emp.CareerAge.eq(year),"pubs_adj"].to_numpy(); b = trajs[year]
    wasserstein.append({"career_age": year,"W1_raw": stats.wasserstein_distance(a,b),"W1_log1p": stats.wasserstein_distance(np.log(a + EPS),np.log(b + EPS))})
pd.DataFrame(wasserstein).to_csv(OUTPUT_DIR / "arw4_wasserstein_by_year.csv",index=False)

summary = {"empirical_mean_y0": emp.loc[emp.CareerAge.eq(0),"pubs_adj"].mean(),"empirical_mean_y20": emp.loc[emp.CareerAge.eq(20),"pubs_adj"].mean(),"simulated_mean_y0": trajs[0].mean(),"simulated_mean_y20": trajs[20].mean(),"ks_year_max_stat": ks_rows[0]["statistic"],"ks_cum_y5_stat": ks_rows[1]["statistic"]}
pd.DataFrame([summary]).to_csv(OUTPUT_DIR / "aw4_diagnostic_summary.csv",index=False)
(OUTPUT_DIR / "arw4_diagnose_manifest.json").write_text(json.dumps({"model": MODEL_NAME,"model_tag": MODEL_TAG,"created_utc": datetime.now(timezone.utc).isoformat(),"empirical_complete": emp_trajs.shape[1],"simulated": trajs.shape[1],"simulation_manifest": str(SIM_MANIFEST_PATH)},indent=2))

print(f"Outputs: {OUTPUT_DIR}")
display(pd.DataFrame(ks_rows))

Outputs: /Users/samlunemagid/Desktop/mean_reversion_repo/dx/output


,diagnostic,statistic,pvalue,n_empirical,n_simulated
0,year_of_maximum,0.065340,8.228905e-03,681,10000
1,cumulative_through_year5,0.115017,8.351958e-08,681,10000
